## Climate TRACE API v7 Examples

### Welcome!

This notebook provides examples of how to use the Climate TRACE API v7. The goal is to cover the most common usage scenarios and over time it may be updated to include additional scenarios.

### Documentation

The complete API schema is available at https://api.climatetrace.org/v7. There you will find all endpoints and parameters currently available in the API. We strive to keep the documentation accurate and up to date. Although new features may be added, we will not make any breaking changes to this version.

### Migrating to v7

If you have used a previous version of the API, you should switch to using v7. It provides all the functionality of v6 and more. You may find that some endpoints and query string parameters that were present in v6 have been removed or renamed in this version. v6 is no longer updated and will soon be deprecated. If you have any difficulty migrating to v7, please use our contact form to send us questions.

### Usage

Please keep in mind that other people rely on this API. Excessive request volumes by one user can negatively impact other users. We ask that you limit the concurrency of your requests to a reasonable level. If you wish to download the entire Climate TRACE dataset, consider using our data download packages at https://climatetrace.org/data or contact us about your data needs.

### About Us

Climate TRACE is a non-profit coalition of organizations building a timely, open, and accessible inventory of exactly where greenhouse gas emissions are coming from.

### Questions?

Message us here: https://climatetrace.org/contact



## Getting Started

Set up the convenience function used by the code in this notebook:

In [ ]:
import requests

BASE_URL = "https://api.climatetrace.org/v7"

def call_api(endpoint, params=None):
    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, params=params)
    print(f"GET {url}")
    print("Status Code:", response.status_code)
    try:
        return response.json()
    except:
        return response.text


## Aggregate Emissions

The [sources/emissions](https://api.climatetrace.org/v7/docs/index.html#tag/emissions-sources/GET/v7/sources/emissions) endpoint is used to retrieve aggregated emissions data. The only required parameters are `year` and `gas`. If these are omitted, default values (`2025` and `co2e_100yr`) are used. In the response, emissions are grouped by calendar month and also aggregated for the year.




In [ ]:
# global methane emissions for 2025
global_2025_emissions = call_api('/sources/emissions', params={'gas': 'ch4', 'year': 2025})
global_2025_emissions

GET https://api.climatetrace.org/v7/sources/emissions
Status Code: 200


{'location': {'name': 'Global'},
 'totals': {'summaries': [{'gas': 'ch4',
    'emissionsQuantity': 411529850.9582714,
    'percentage': 100}],
  'timeseries': [{'year': 2025,
    'month': 1,
    'gas': 'ch4',
    'emissionsQuantity': 33708234.818669945},
   {'year': 2025,
    'month': 2,
    'gas': 'ch4',
    'emissionsQuantity': 33019914.649375774},
   {'year': 2025,
    'month': 3,
    'gas': 'ch4',
    'emissionsQuantity': 33068777.941748027},
   {'year': 2025,
    'month': 4,
    'gas': 'ch4',
    'emissionsQuantity': 33490331.45467559},
   {'year': 2025,
    'month': 5,
    'gas': 'ch4',
    'emissionsQuantity': 34399943.28042569},
   {'year': 2025,
    'month': 6,
    'gas': 'ch4',
    'emissionsQuantity': 35060195.01824131},
   {'year': 2025,
    'month': 7,
    'gas': 'ch4',
    'emissionsQuantity': 36091128.493436426},
   {'year': 2025,
    'month': 8,
    'gas': 'ch4',
    'emissionsQuantity': 36003065.31348097},
   {'year': 2025,
    'month': 9,
    'gas': 'ch4',
    'emissi

### Filter Parameters

Emissions can be filtered further in several ways:

* Subsectors
* Sectors (the parent group of Subsectors)
* Location
  * Continent
  * Country group (e.g. G7)
  * Country/Nation (GADM / Administrative Area level 0)
  * State/Province (GADM / Administrative Area level 1)
  * County/District (GADM / Administrative Area level 2)
  * City (technically a functional urban area)
* Source Owners

These filters correspond to standardized query string parameters that are documented [here](https://api.climatetrace.org/v7/docs/index.html#tag/emissions-sources/GET/v7/sources/emissions).


In [ ]:
# North America methane emissions from agriculture
na_ch4_2025_ag_emissions = call_api('/sources/emissions', params={
    'gas': 'ch4',
    'year': 2025,
    'sectors': 'agriculture',
    'continent': 'North America'
  })
na_ch4_2025_ag_emissions

GET https://api.climatetrace.org/v7/sources/emissions
Status Code: 200


{'location': {'name': 'North America', 'continent': 'North America'},
 'totals': {'summaries': [{'gas': 'ch4',
    'emissionsQuantity': 16248354.037331173,
    'percentage': 100}],
  'timeseries': [{'year': 2025,
    'month': 1,
    'gas': 'ch4',
    'emissionsQuantity': 1283819.585921951},
   {'year': 2025,
    'month': 2,
    'gas': 'ch4',
    'emissionsQuantity': 1288343.7221826015},
   {'year': 2025,
    'month': 3,
    'gas': 'ch4',
    'emissionsQuantity': 1302350.435471294},
   {'year': 2025,
    'month': 4,
    'gas': 'ch4',
    'emissionsQuantity': 1385437.1828514042},
   {'year': 2025,
    'month': 5,
    'gas': 'ch4',
    'emissionsQuantity': 1401948.6033730733},
   {'year': 2025,
    'month': 6,
    'gas': 'ch4',
    'emissionsQuantity': 1398894.043727183},
   {'year': 2025,
    'month': 7,
    'gas': 'ch4',
    'emissionsQuantity': 1424488.1711098398},
   {'year': 2025,
    'month': 8,
    'gas': 'ch4',
    'emissionsQuantity': 1425442.379130988},
   {'year': 2025,
    'mo

## Countries

v7 retains the all-important ability to filter data by country. Unlike v6, this filter is consolidated under a query string parameter: `gadmId`.

Climate TRACE uses the [GADM](https://gadm.org/) system for administrative area boundary data in which countries are "level 0" administrative areas. The `gadmId` for a country is its 3-letter country code (e.g. United States --> `USA`, Brazil --> `BRA`, Kenya --> `KEN`, etc). The attribute names `id` and `gadmId` may be interchangeable depending on the context.

Use the [/admins](https://api.climatetrace.org/v7/docs/index.html#tag/administrative-areas/GET/v7/admins) to search for a country by name and get its `id` or `gadmId`.

In [ ]:
# search for Denmark by name to get its id (aka gadmId)
denmark_search_results = call_api('/admins', params={'name': 'Denmark', 'level': 0, 'limit': '5', 'offset': '0'})
denmark_search_results

GET https://api.climatetrace.org/v7/admins
Status Code: 200


[{'id': 'DNK',
  'name': 'Denmark',
  'full_name': 'Denmark (DNK)',
  'level': 0,
  'level_0_id': 'DNK',
  'level_1_id': '',
  'level_2_id': ''}]

In [ ]:
# search returns an array even if there was just 1 result
denmark = denmark_search_results[0]
denmark

{'id': 'DNK',
 'name': 'Denmark',
 'full_name': 'Denmark (DNK)',
 'level': 0,
 'level_0_id': 'DNK',
 'level_1_id': '',
 'level_2_id': ''}

Alternatively, you can loop through a list of all countries using the [definitions/countries](https://api.climatetrace.org/v7/docs/index.html#tag/definitions/get/v7/definitions/countries) endpoint.

In [ ]:
countries_list = call_api('/definitions/countries')
countries_list[0:10]

GET https://api.climatetrace.org/v7/definitions/countries
Status Code: 200


[{'id': 'ABW', 'name': 'Aruba', 'continent': 'North America'},
 {'id': 'AFG', 'name': 'Afghanistan', 'continent': 'Asia'},
 {'id': 'AGO', 'name': 'Angola', 'continent': 'Africa'},
 {'id': 'AIA', 'name': 'Anguilla', 'continent': 'North America'},
 {'id': 'ALA', 'name': 'Åland Islands', 'continent': 'Europe'},
 {'id': 'ALB', 'name': 'Albania', 'continent': 'Europe'},
 {'id': 'AND', 'name': 'Andorra', 'continent': 'Europe'},
 {'id': 'ARE', 'name': 'United Arab Emirates', 'continent': 'Asia'},
 {'id': 'ARG', 'name': 'Argentina', 'continent': 'South America'},
 {'id': 'ARM', 'name': 'Armenia', 'continent': 'Asia'}]

## Country Emissions

Once you know the `gadmId` for your desired country, you can get its aggregated emissions using the [sources/emissions](https://api.climatetrace.org/v7/docs/index.html#tag/emissions-sources/GET/v7/sources/emissions) endpoint.

In [ ]:
all_denmark_co2_emissions_2023 = call_api("/sources/emissions", params={"gadmId": denmark["id"], "year": 2023, "gas": "co2"})
all_denmark_co2_emissions_2023

GET https://api.climatetrace.org/v7/sources/emissions
Status Code: 200


{'location': {'name': 'Denmark', 'gadmId': 'DNK', 'country': 'DNK'},
 'totals': {'summaries': [{'gas': 'co2',
    'emissionsQuantity': 39755082.902364776,
    'percentage': 100}],
  'timeseries': [{'year': 2023,
    'month': 1,
    'gas': 'co2',
    'emissionsQuantity': 3132576.805323715},
   {'year': 2023,
    'month': 2,
    'gas': 'co2',
    'emissionsQuantity': 2979172.449732134},
   {'year': 2023,
    'month': 3,
    'gas': 'co2',
    'emissionsQuantity': 3390187.565039851},
   {'year': 2023,
    'month': 4,
    'gas': 'co2',
    'emissionsQuantity': 3602455.9810981504},
   {'year': 2023,
    'month': 5,
    'gas': 'co2',
    'emissionsQuantity': 3742023.330717811},
   {'year': 2023,
    'month': 6,
    'gas': 'co2',
    'emissionsQuantity': 3618163.336070507},
   {'year': 2023,
    'month': 7,
    'gas': 'co2',
    'emissionsQuantity': 3460350.77273439},
   {'year': 2023,
    'month': 8,
    'gas': 'co2',
    'emissionsQuantity': 3378656.23557985},
   {'year': 2023,
    'month': 

Again, the `sectors` and `subsectors` parameters can filter aggregated country emissions further.

In [ ]:
sector = "manufacturing"
year = 2023
gas = "co2"
manufacturing_denmark_co2_emissions_2023 = call_api("/sources/emissions", params={"gadmId": denmark["id"], "year": year, "gas": gas, "sectors": sector})
try:
  emissions_quantity = manufacturing_denmark_co2_emissions_2023['totals']['summaries'][0]['emissionsQuantity']
  print(f"denmark's {sector} sector(s) emitted {emissions_quantity} tonnes of {gas} in {year}")
except Exception as e:
  print(f"error: {e}")

GET https://api.climatetrace.org/v7/sources/emissions
Status Code: 200
denmark's manufacturing sector(s) emitted 3669976.4426967874 tonnes of co2 in 2023


## Sources of Emissions

For more granular detail about where exactly emissions come from, you can utilize Climate TRACE's data about sources of emissions. The [/sources](https://api.climatetrace.org/v7/docs/index.html#tag/emissions-sources/GET/v7/sources) endpoint is for searching for sources (previously known as "assets").

In [ ]:
# sources of fossil fuel emissions in Denmark
sources_mfg_denmark_2023 = call_api("/sources", params={"gadmId": denmark["id"], "year": year, "gas": gas, "sectors": sector})
sources_mfg_denmark_2023

GET https://api.climatetrace.org/v7/sources
Status Code: 200


[{'id': 1896644,
  'name': 'Aalborg Cement Plant',
  'sector': 'manufacturing',
  'subsector': 'cement',
  'country': 'DNK',
  'assetType': 'integrated dry',
  'sourceType': 'point-source',
  'centroid': {'longitude': 9.9761291643243,
   'latitude': 57.0625384389328,
   'srid': 4326},
  'gas': 'co2',
  'emissionsQuantity': 766223.27566,
  'emissionsFactor': 0.4315141999705672,
  'emissionsFactorUnits': 't of CO2 per t of cement',
  'activity': 1775661.7875199998,
  'activityUnits': 't of cement',
  'capacity': 2363000.0002,
  'capacityUnits': 't of cement',
  'capacityFactor': 0.7514438372279776,
  'year': 2023},
 {'id': 38480349,
  'name': 'Nordic Sugar A/S Nykøbing',
  'sector': 'manufacturing',
  'subsector': 'food-beverage-tobacco',
  'country': 'DNK',
  'assetType': 'Animal And Vegetable Products From The Food And Beverage Sector',
  'sourceType': 'point-source',
  'centroid': {'longitude': 11.874207, 'latitude': 54.763005, 'srid': 4326},
  'gas': 'co2',
  'emissionsQuantity': 105